# US Stocks Volatility Analysis - Version 3: Risk Metrics & Performance\n\n**Author:** Trade Vectors LLP\n**Date:** March 2026\n\n## Overview\nThis notebook focuses on risk management applications of volatility analysis:\n- Value at Risk (VaR) Calculation\n- Expected Shortfall (Conditional VaR)\n- Volatility-Adjusted Returns (Sharpe & Sortino Ratios)\n- Maximum Drawdown Analysis\n\n## Data Source\nHistorical stock price data (NVDA, AAPL, MSFT, AMZN, IBM) from the USStocks folder.

In [ ]:
# Import libraries\nimport pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom scipy.stats import norm, t\nimport warnings\nwarnings.filterwarnings('ignore')\n\nprint('Libraries loaded successfully')

In [ ]:
# Load and prepare data\ndef load_stock_data(ticker):\n    df = pd.read_csv(f'../USStocks/{ticker}_STK.csv')\n    df['Date'] = pd.to_datetime(df['Date'])\n    df.set_index('Date', inplace=True)\n    df.sort_index(inplace=True)\n    df['Returns'] = df['Close'].pct_change()\n    return df.dropna()\n\ntickers = ['NVDA', 'AAPL', 'MSFT', 'AMZN', 'IBM']\nstock_data = {ticker: load_stock_data(ticker) for ticker in tickers}

In [ ]:
# Calculate Value at Risk (VaR)\ndef calculate_var_es(returns, confidence_level=0.95):\n    """Calculate Historical and Parametric VaR and Expected Shortfall"""\n    # 1. Historical VaR\n    h_var = np.percentile(returns, (1 - confidence_level) * 100)\n    h_es = returns[returns <= h_var].mean()\n    \n    # 2. Parametric VaR (Normal Distribution)\n    mu = returns.mean()\n    sigma = returns.std()\n    p_var = norm.ppf(1 - confidence_level, mu, sigma)\n    p_es = mu - sigma * (norm.pdf(norm.ppf(1 - confidence_level)) / (1 - confidence_level))\n    \n    return {\n        'Hist_VaR': h_var,\n        'Hist_ES': h_es,\n        'Param_VaR': p_var,\n        'Param_ES': p_es\n    }\n\nrisk_metrics = {}\nfor ticker, df in stock_data.items():\n    risk_metrics[ticker] = calculate_var_es(df['Returns'])\n\nrisk_df = pd.DataFrame(risk_metrics).T\nprint('95% Daily Risk Metrics')\nprint(risk_df.style.format('{:.2%}'))

In [ ]:
# Drawdown Analysis\ndef calculate_drawdown(returns):\n    cum_rets = (1 + returns).cumprod()\n    running_max = cum_rets.cummax()\n    drawdown = (cum_rets - running_max) / running_max\n    return drawdown\n\nplt.figure(figsize=(14, 7))\nfor ticker, df in stock_data.items():\n    dd = calculate_drawdown(df['Returns'])\n    plt.plot(dd.index, dd, label=f'{ticker} (Max: {dd.min():.1%})')\n\nplt.title('Historical Drawdown Analysis', fontsize=15)\nplt.ylabel('Drawdown %')\nplt.legend()\nplt.grid(True, alpha=0.3)\nplt.tight_layout()\nplt.show()

In [ ]:
# Performance Metrics Table\nperf_stats = []\nfor ticker, df in stock_data.items():\n    rets = df['Returns']\n    ann_ret = rets.mean() * 252\n    ann_vol = rets.std() * np.sqrt(252)\n    sharpe = ann_ret / ann_vol if ann_vol != 0 else 0\n    \n    perf_stats.append({\n        'Ticker': ticker,\n        'Ann. Return': ann_ret,\n        'Ann. Volatility': ann_vol,\n        'Sharpe Ratio': sharpe,\n        'Max Drawdown': calculate_drawdown(rets).min()\n    })\n\nsummary = pd.DataFrame(perf_stats)\nprint('Historical Performance Summary')\nprint(summary.to_string(index=False))

## Key Takeaways\n1. **VaR**: Quantifies potential losses over a specific time frame\n2. **Expected Shortfall**: Measures average loss beyond VaR (tail risk)\n3. **Risk-Adjusted Return**: Sharpe ratio helps compare assets with different volatility levels\n\n## Next Steps\n- Monte Carlo VaR simulation\n- Extreme Value Theory (EVT) for tail risk\n- Stress testing against historical market crashes\n- Optimal portfolio construction based on volatility